# 04 — RAG Pipeline, LangSmith Tracing & RAGAS Evaluation

**Day 2 | Data Engineering & AI Bootcamp**

RAG (Retrieval Augmented Generation) grounds LLM answers in real documents, reducing hallucinations.
This notebook builds a complete RAG pipeline, adds observability via LangSmith,
and evaluates quality using RAGAS metrics.

**All core demos work without API keys.** LangSmith and real RAGAS are opt-in via `.env`.

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from day2.rag import (
    Document, RAGPipeline, LangSmithTracer,
    ragas_proxy_metrics, record_baseline,
    SAMPLE_DOCUMENTS, EVAL_QUESTIONS,
    chunk_document, chunk_corpus,
)

sns.set_theme(style='whitegrid')
print('Setup complete ✓')
print(f'LangSmith enabled: {bool(os.getenv("LANGCHAIN_API_KEY"))}')
print(f'OpenAI (RAGAS):    {bool(os.getenv("OPENAI_API_KEY"))}')

## 1. RAG Architecture

RAG solves two LLM problems: **knowledge cutoff** and **hallucination**.
Instead of relying on what the model memorised during training, RAG retrieves relevant
information at query time and puts it directly into the prompt as context.

```
INDEXING (offline):    Documents → Chunk → Embed → Vector DB
QUERYING (real-time):  Question → Embed → Retrieve → [Context + Question] → LLM → Answer
```

> **Real world:** JP Morgan uses RAG so analysts can ask questions about internal research.
> Shell uses RAG for safety manual Q&A. Amdocs uses it for customer support automation.
> The vector DB retrieves the relevant paragraphs; the LLM synthesises the answer.

In [ ]:
# Show the RAG pipeline architecture
print('RAG Pipeline Architecture:')
print()
print('INDEXING PHASE (run once / nightly):')
print('  1. Load documents  →  2. Chunk (200 words, 40 overlap)')
print('  3. Embed chunks    →  4. Store in vector DB with metadata')
print()
print('RETRIEVAL PHASE (per query, real-time):')
print('  1. Embed user question')
print('  2. Cosine similarity search → top-k chunks')
print('  3. Build prompt: [System] + [Chunk 1] + [Chunk 2] + ... + [Question]')
print('  4. Call LLM → get answer')
print('  5. Log trace to LangSmith (query, context, answer, latency)')
print()
print('EVALUATION (periodic):')
print('  RAGAS: faithfulness, answer_relevancy, context_precision, context_recall')

# Show chunking
long_doc = (
    'Machine learning is a subset of artificial intelligence that enables systems to learn '
    'from data. Supervised learning uses labelled examples to train models. Unsupervised learning '
    'finds hidden patterns without labels. Reinforcement learning trains agents through rewards. '
    'Deep learning uses neural networks with many layers. Transfer learning applies pre-trained '
    'models to new tasks. Natural language processing handles text understanding and generation.'
)

chunks = chunk_document(long_doc, chunk_size=20, overlap=5)
print(f'\nChunking demo (chunk_size=20 words, overlap=5):')
print(f'  Original: {len(long_doc.split())} words')
print(f'  Chunks:   {len(chunks)}')
for i, c in enumerate(chunks):
    print(f'  Chunk {i}: "{c[:70]}..."')

## 2. LLM Observability — Why Trace Every Call?

LLM observability means recording every LLM interaction: input, output, latency, cost.
Without tracing, debugging a bad answer is impossible — you don't know what context was retrieved,
what prompt was sent, or how long it took.

**LangSmith** is Anthropic/LangChain's observability platform:
- Trace every LLM call with input/output/latency
- Create datasets of good/bad examples for fine-tuning
- Run offline experiments to compare prompt changes
- Share traces with team for debugging

> **Without LANGCHAIN_API_KEY:** traces are saved to a local JSON file — same data, local storage.
> This is how you develop and review before connecting to the cloud platform.

In [ ]:
# Setup tracer (works with or without API key)
tracer = LangSmithTracer(project_name='day2-rag-demo')

# Manually log a sample trace
trace_id = tracer.log_rag_call(
    query   = 'What is supervised learning?',
    context = 'Supervised learning uses labelled training examples.',
    answer  = 'Supervised learning is a type of ML that learns from labelled data.',
    latency = 0.342,
    metadata = {'model': 'all-MiniLM-L6-v2', 'top_k': 3},
)

print(f'Logged trace: {trace_id}')

# Show what a trace looks like
traces = tracer.get_local_traces()
if traces:
    print(f'\nTrace entry:')
    print(json.dumps(traces[0], indent=2))

print()
print('In LangSmith cloud, you would see:')
print('  - Visual timeline of the retrieval + generation steps')
print('  - Latency breakdown per step')
print('  - Full prompt sent to LLM (with context)')
print('  - Token counts and estimated cost')
print('  - Ability to annotate good/bad examples')

## 3. Building the RAG Pipeline End-to-End

Now we build a complete RAG pipeline: index documents, then answer questions.
The pipeline uses sentence-transformers for embeddings and ChromaDB for retrieval.
The 'LLM' in this demo is a mock function — in production, replace with OpenAI/Claude/Databricks.

> **Replacing the mock LLM:**
> ```python
> import anthropic
> client = anthropic.Anthropic()
> def real_llm(prompt, context):
>     response = client.messages.create(
>         model='claude-sonnet-4-6',
>         max_tokens=512,
>         messages=[{'role': 'user', 'content': f'Context: {context}\n\nQuestion: {prompt}'}]
>     )
>     return response.content[0].text
> ```

In [ ]:
# Build pipeline
pipeline = RAGPipeline(top_k=3, tracer=tracer)

# Index documents
print('[Indexing] Loading sample documents...')
pipeline.index(SAMPLE_DOCUMENTS)

# Query the pipeline
print('\n[Querying]')
for question, _ in EVAL_QUESTIONS:
    result = pipeline.query(question)
    print(f'\nQ: {question}')
    print(f'A: {result.answer[:100]}...')
    print(f'   Retrieved {result.num_chunks} chunks | Latency: {result.latency_ms:.1f}ms | Trace: {result.trace_id}')
    for chunk in result.retrieved_chunks:
        print(f'   [{chunk.score:.4f}] {chunk.text[:60]}')

## 4. RAGAS Evaluation Metrics

RAGAS provides 4 core metrics to measure RAG quality:

| Metric | Question Answered | Range |
|---|---|---|
| **Faithfulness** | Is the answer supported by the retrieved context? | 0–1 |
| **Answer Relevancy** | Does the answer directly address the question? | 0–1 |
| **Context Precision** | Are the top-k chunks relevant? (signal/noise) | 0–1 |
| **Context Recall** | Did we retrieve all info needed to answer? | 0–1 |

> **With `OPENAI_API_KEY`:** real RAGAS uses GPT-4 as a judge for accurate scores.
> **Without key:** we use proxy metrics (TF-IDF overlap + embedding similarity) — same concept.
> Use the real metrics for production baselines; proxy metrics for fast dev iteration.

In [ ]:
# Evaluate all questions
eval_results = []

print('Running RAGAS evaluation (proxy metrics)...\n')
print(f'{"Question":<50} {"Faith":>7} {"Relev":>7} {"Prec":>7} {"Recall":>7} {"Avg":>7}')
print('─' * 90)

for question, ground_truth in EVAL_QUESTIONS:
    result  = pipeline.query(question)
    metrics = ragas_proxy_metrics(result, ground_truth)
    eval_results.append(metrics)

    print(f'{question[:50]:<50} '
          f'{metrics.faithfulness:>7.4f} '
          f'{metrics.answer_relevancy:>7.4f} '
          f'{metrics.context_precision:>7.4f} '
          f'{metrics.context_recall:>7.4f} '
          f'{metrics.average:>7.4f}')

avg_scores = {
    'faithfulness':      np.mean([r.faithfulness      for r in eval_results]),
    'answer_relevancy':  np.mean([r.answer_relevancy   for r in eval_results]),
    'context_precision': np.mean([r.context_precision  for r in eval_results]),
    'context_recall':    np.mean([r.context_recall     for r in eval_results]),
}

print('─' * 90)
print(f'{"AVERAGE":<50} '
      f'{avg_scores["faithfulness"]:>7.4f} '
      f'{avg_scores["answer_relevancy"]:>7.4f} '
      f'{avg_scores["context_precision"]:>7.4f} '
      f'{avg_scores["context_recall"]:>7.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Radar-style bar for average scores
metric_names = list(avg_scores.keys())
metric_vals  = list(avg_scores.values())
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

bars = axes[0].barh(metric_names, metric_vals, color=colors, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, metric_vals):
    axes[0].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=11, fontweight='bold')
axes[0].set_xlim(0, 1.2)
axes[0].axvline(0.7, color='gray', linestyle='--', alpha=0.5, label='Acceptable (0.70)')
axes[0].axvline(0.9, color='green', linestyle='--', alpha=0.5, label='Good (0.90)')
axes[0].legend(fontsize=9)
axes[0].set_title('RAGAS Metrics — Average Scores', fontweight='bold')

# Per-question breakdown
q_labels = [f'Q{i+1}' for i in range(len(eval_results))]
x = np.arange(len(q_labels))
width = 0.2

for j, (metric, color) in enumerate(zip(metric_names, colors)):
    vals = [getattr(r, metric) for r in eval_results]
    axes[1].bar(x + j * width, vals, width, label=metric, color=color, alpha=0.85)

axes[1].set_xticks(x + width * 1.5)
axes[1].set_xticklabels(q_labels)
axes[1].set_ylim(0, 1.2)
axes[1].axhline(0.7, color='gray', linestyle='--', alpha=0.5)
axes[1].legend(fontsize=8)
axes[1].set_title('RAGAS Metrics — Per Question', fontweight='bold')
axes[1].set_ylabel('Score')

plt.suptitle('Day 2 RAG Baseline Evaluation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Recording the Baseline

A baseline is the **starting point** — the score you compare all future improvements against.
Save it to a JSON file, commit to git, and track over time.
Every model upgrade, chunking strategy change, or prompt change should be compared to baseline.

> **Day 2 baseline → Day 3+ goal:** Can we improve faithfulness > 0.90 by using better chunking?
> Can we improve context precision by adding metadata filtering?
> Systematic improvement = measure baseline → change one thing → measure again.

In [ ]:
baseline = record_baseline(eval_results, 'all-MiniLM-L6-v2', output_path='rag_baseline.json')

print('Baseline saved to rag_baseline.json')
print(f'\nOverall mean score: {baseline["overall_mean"]:.4f}')
print(f'\nMetric breakdown:')
for metric, stats in baseline['metrics'].items():
    bar = '█' * int(stats['mean'] * 20)
    print(f'  {metric:<22}: {stats["mean"]:.4f} ± {stats["std"]:.4f}  {bar}')

# Show the saved file content
print(f'\nBaseline JSON:')
print(json.dumps(baseline, indent=2))

## 6. Reproducibility Script

A baseline is only useful if it can be reproduced.
Fix all random seeds, pin the model name, pin the eval questions.
Anyone on the team should be able to run this and get the same numbers.

> **In production:** pin the model hash (not just the name), log the ChromaDB index version,
> and record the git commit hash of the evaluation code. MLflow or Databricks Experiments
> do this automatically when you call `mlflow.log_metrics()`.

In [ ]:
def run_reproducible_eval(
    documents: list,
    eval_questions: list,
    embedding_model: str = 'all-MiniLM-L6-v2',
    top_k: int = 3,
    seed: int = 42,
) -> dict:
    """
    Fully reproducible RAG evaluation.
    Returns baseline metrics dict — identical output every run.
    """
    np.random.seed(seed)

    pipeline = RAGPipeline(embedding_model=embedding_model, top_k=top_k)
    pipeline.index(documents)

    evals = []
    for question, ground_truth in eval_questions:
        result  = pipeline.query(question)
        metrics = ragas_proxy_metrics(result, ground_truth)
        evals.append(metrics)

    return record_baseline(evals, embedding_model, output_path='rag_baseline_repro.json')

# Run twice — should produce identical scores
b1 = run_reproducible_eval(SAMPLE_DOCUMENTS, EVAL_QUESTIONS)
b2 = run_reproducible_eval(SAMPLE_DOCUMENTS, EVAL_QUESTIONS)

print(f'\nRun 1 overall: {b1["overall_mean"]:.6f}')
print(f'Run 2 overall: {b2["overall_mean"]:.6f}')
print(f'Reproducible:  {abs(b1["overall_mean"] - b2["overall_mean"]) < 1e-6}')

# Print traces captured
traces = tracer.get_local_traces()
print(f'\nTotal traces captured: {len(traces)}')
tracer.save_local_traces('traces.json')